# react-lm: Step 3 (Chat Template) + Step 4 (QLoRA)

Fine-tune **Qwen2.5-Coder-3B-Instruct** on the React golden dataset from [react-lm](https://github.com/your-org/react-lm).

**Prerequisites**
- Run `python check_step2_ready.py` locally (Step 2 must pass).
- Upload `train.jsonl` below, or clone this repo so `train.jsonl` is on disk.
- Colab runtime: **GPU** (T4 is enough).

Step 3 applies Unsloth's Qwen2.5 chat template via `apply_chat_template()` — do **not** hand-format `<|im_start|>` tokens.

See [PLAN.md](https://github.com/your-org/react-lm/blob/main/PLAN.md) for eval (Step 5), GGUF export (Step 6), and Ollama (Step 7).

## 1. Install Unsloth (GPU required)

In [ ]:
%%capture
import os, re, subprocess
if "COLAB_RELEASE_TAG" in os.environ:
    subprocess.run(
        "pip install -q unsloth datasets transformers trl accelerate bitsandbytes",
        shell=True,
        check=False,
    )
else:
    print("Local run: pip install unsloth datasets transformers trl accelerate bitsandbytes")

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get `train.jsonl`

In [ ]:
from pathlib import Path
import os

TRAIN_PATH = Path("train.jsonl")
REPO_URL = os.environ.get("REACT_LM_REPO", "")  # optional: set to your git remote

if TRAIN_PATH.is_file():
    print(f"Using {TRAIN_PATH.resolve()} ({TRAIN_PATH.stat().st_size // 1024} KiB)")
elif os.environ.get("COLAB_RELEASE_TAG"):
    print("Option A: upload train.jsonl")
    from google.colab import files

    uploaded = files.upload()
    if "train.jsonl" not in uploaded:
        raise FileNotFoundError("Upload a file named train.jsonl")
else:
    raise FileNotFoundError(
        "train.jsonl not found. In Colab: upload the file. "
        "Locally: run this notebook from the react-lm repo root."
    )

assert TRAIN_PATH.is_file()
print("Ready:", TRAIN_PATH.resolve())

## 3. Load model + Qwen2.5 chat template (Step 3)

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

print("Model:", MODEL_NAME)
print("Chat template: qwen-2.5")

## 4. Map `conversations` → `text` (Step 3)

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files=str(TRAIN_PATH), split="train")
print("Rows loaded:", len(raw))
assert "conversations" in raw.column_names, "Expected conversations column in train.jsonl"


def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in convos
    ]
    return {"text": texts}


dataset = raw.map(formatting_prompts_func, batched=True, remove_columns=raw.column_names)
print("Mapped to text column:", dataset)

## 5. Verify formatted sample (Step 3)

In [ ]:
sample_text = dataset[0]["text"]
print(sample_text[:2000])
print("\n---")
print("rows:", len(dataset))

for marker in ("<|im_start|>system", "<|im_start|>user", "<|im_start|>assistant"):
    assert marker in sample_text, f"Missing {marker} in formatted text"

lengths = []
for i in range(min(20, len(dataset))):
    enc = tokenizer(dataset[i]["text"], return_length=True)
    lengths.append(int(enc["length"][0]))

print(f"Token lengths (first {len(lengths)} rows): min={min(lengths)} max={max(lengths)} mean={sum(lengths)/len(lengths):.0f}")
over = sum(1 for L in lengths if L > MAX_SEQ_LENGTH)
if over:
    print(f"WARNING: {over} of {len(lengths)} sampled rows exceed MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}")
else:
    print(f"OK: sampled rows are under {MAX_SEQ_LENGTH} tokens")

## 6. QLoRA adapters (Step 4)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

## 7. Train with SFTTrainer (Step 4)

Adjust `num_train_epochs`, `max_steps`, and batch size for your GPU. Full run is typically **2–5 hours** on a T4.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "react-expert-lora"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

## 8. Save adapter + optional GGUF (Steps 6–7)

- Evaluate on held-out `eval.jsonl` **before** export (Step 5 in PLAN.md).
- Use the repo `Modelfile` when registering with Ollama.

In [ ]:
SAVE_DIR = "react-expert"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved LoRA adapter to {SAVE_DIR}/")

# Uncomment after eval passes (Step 6):
# model.save_pretrained_gguf(SAVE_DIR, tokenizer, quantization_method="q4_k_m")
# Then download react-expert.Q4_K_M.gguf and use the repo Modelfile with Ollama.